# Exploration 02 — data cleaning and feature engineering (X2: molecule and atom level)

> **Exploratory** (CLAUDE.md, "Exploration"). **Data:** the development universe: the training pool P plus the two
> development sets (`dev`, `dev_unseen`). No test molecule is read. Features are computed from Z and R only (plus
> the DFT block, exploration-only and labeled). The cleaning rules are **unsupervised** and fit on P: they look at
> the features, never at |μ|.

**Part 2 of the classical analysis.** Input: the EDA's findings (`results/explore01_recommendations.csv`). Output,
used by every later notebook:
- `data/processed/explore_features.parquet`: one row per development-universe molecule, every kept molecule-level
  feature, the target and bookkeeping (regenerated, not committed);
- `results/explore02_feature_catalog.csv`: one row per feature, molecule- and atom-level, with its block, whether it
  is headline-legal, whether it was kept, and why it was dropped if not.

Per-atom features are not stored (about 0.7 GB); `explore.load_atoms` recomputes them from Z and R in about a minute
and keeps the columns this notebook keeps. Thresholds come from `configs/explore.yaml`.

**What changed from X1.** X1 described each molecule by global, rotation-invariant summaries: atom counts, the sorted
Coulomb-matrix spectrum and engineered polarity proxies. Its five model families all stalled at the same error,
because such summaries discard the directions that decide whether bond dipoles add or cancel. X2 adds the atom as a
unit (`atoms.py`): each atom's environment, from which notebook 04's latent-charge model predicts a charge, and
molecule-level blocks built from those environments (charged groups, the QEq dipole, radial distributions).

In [ ]:
# Environment check: stops here, with setup instructions, unless this kernel is the project
# environment (README.md, "Setup"). Version differences from requirements.txt only warn.
try:
    from qm9dipole.provenance import check_environment
except ModuleNotFoundError as e:
    raise RuntimeError(
        "The qm9dipole package is not installed in this kernel's environment. Follow "
        "README.md (Setup), then select that environment as the notebook kernel."
    ) from e

print(check_environment())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from qm9dipole import REPO_ROOT
from qm9dipole.data import load_qm9_table
from qm9dipole.explore import (ATOM_BLOCK, CATALOG_NAME, DFT_BLOCKS, FEATURES_PATH, LEGAL_BLOCKS, development_ids,
                               feature_sets, load_config)
from qm9dipole.plots import INK_SECONDARY, MUTED, ROLE_COLORS, SERIES, SURFACE, use_style
from qm9dipole.provenance import RESULTS_DIR, save_result
from qm9dipole.splits import load_splits

use_style()
FIG_DIR = REPO_ROOT / "figures"
CFG = load_config()

table = load_qm9_table()
splits = load_splits()
ids = development_ids(splits)
U = table.set_index("id").loc[ids].reset_index()
mol = U.set_index("id")
subset = pd.Series("pool", index=mol.index, name="subset")
subset.loc[splits.dev], subset.loc[splits.dev_unseen] = "dev", "dev_unseen"
tests = set(splits.test_unseen) | set(splits.test_familiar)
assert not set(ids) & tests, "a test molecule entered the development universe"
in_pool = (subset == "pool").to_numpy()
y = mol["mu"]
print("development universe:", subset.value_counts().to_dict(), f"({len(U):,} molecules, {U['n_atoms'].sum():,} atoms)")

recommendations = pd.read_csv(RESULTS_DIR / "explore01_recommendations.csv")
print("\nWhat the EDA (notebook 01) asks this notebook to do:")
display(recommendations[recommendations["notebook"].astype(str).str.contains("02")].drop(columns="notebook")
        .style.hide(axis="index"))

## 1. Record-level cleaning

### 1a. Doubled frequency lists (exploration-only fields)
Some raw records print their frequency list twice, with identical halves. Keep the first copy.

In [ ]:
from qm9dipole.cleaning import clean_frequencies, is_linear, vibrational_modes

freqs = pd.Series([clean_frequencies(f, n) for f, n in zip(mol["freqs"], mol["n_atoms"])], index=mol.index)
changed = mol["freqs"].map(len) != freqs.map(len)
linear = pd.Series([is_linear(f, n) for f, n in zip(freqs, mol["n_atoms"])], index=mol.index)
assert all(len(f) == vibrational_modes(n, lin) for f, n, lin in zip(freqs, mol["n_atoms"], linear))
print(f"de-doubled: {changed.sum()} lists; every molecule now lists exactly 3n − 6 modes (3n − 5 for the "
      f"{linear.sum()} linear ones)")

### 1b. Rotational constants → moments of inertia
A, B, C = 505.379 GHz·amu·Å² / I, with I computed from Z and R (EDA §2). The moments carry the same information, are
finite for linear molecules, and are far less skewed. They enter as engineered features computed from Z and R;
QM9's A, B, C columns are not used.

In [ ]:
from qm9dipole.cleaning import moments_from_rotational_constants
from qm9dipole.features import inertia_moments

from_qm9 = np.sort(moments_from_rotational_constants(mol[["A", "B", "C"]].to_numpy()), axis=1)
from_geometry = np.array([inertia_moments(z, r) for z, r in zip(mol["Z"], mol["R"])])
rel = np.abs(from_geometry - from_qm9).max(axis=1) / from_geometry.max(axis=1)
print(f"moments from Z, R vs from QM9's A, B, C: max relative difference {rel.max():.1e}; skewness of A "
      f"{mol['A'].skew():.0f} vs the smallest moment {pd.Series(from_geometry[:, 0]).skew():.1f}")

### 1c. Duplicates
Removed before splitting (notebook 01): no molecule appears twice in the development universe, so every row below is
a distinct molecule and no CV fold can see a validation molecule's twin.

## 2. Feature engineering

| Block | Level | Features | Source |
|---|---|---|---|
| `composition` | molecule | counts of C, H, N, O, F | Z |
| `cm` | molecule | 29 Coulomb-matrix eigenvalues (M2) | Z, R |
| `engineered` | molecule | X1's geometry, bond-dipole proxies, bond counts, bond orders, rings, moments of inertia | Z, R + tables |
| `groups` (new) | molecule | atom types (element × number of bonds), ring atoms, ammonium, carboxylate, nitro-like, zwitterion, planar/pyramidal N | Z, R |
| `qeq` (new) | molecule | charge-equilibration dipole (D), largest charge, charge separation | Z, R + QEq parameters |
| `rdf` (new) | molecule | radial distribution per element pair (15 pairs × 8 distances) | Z, R |
| `atom` (new) | **atom** | 93 per-atom environment features (`atoms.ATOM_FEATURE_NAMES`): the latent-charge model's inputs | Z, R + tables |
| `dft_property` | molecule | α, HOMO, LUMO, gap, ⟨R²⟩, ZPVE, U0, U, H, G, Cv | DFT, exploration only |
| `charge` | molecule | Mulliken point-charge dipole (D), max \|q\|, charge separation | DFT, exploration only |
| `frequency` | molecule | lowest/highest/mean mode, X–H stretches, soft modes | DFT, exploration only |

In [ ]:
import time

from qm9dipole.atoms import ATOM_FEATURE_NAMES, GROUP_NAMES, QEQ_NAMES, RDF_NAMES, build_atom_data
from qm9dipole.descriptors import feature_frame
from qm9dipole.eda import DATA_DICTIONARY
from qm9dipole.features import (CHARGE_FEATURE_NAMES, FREQUENCY_FEATURE_NAMES, charge_features,
                                frequency_features)

start = time.perf_counter()
atoms_U, atom_blocks = build_atom_data(U[["id", "Z", "R"]])
DFT_PROPERTIES = ["alpha", "homo", "lumo", "gap", "r2", "zpve", "U0", "U", "H", "G", "Cv"]
blocks = {
    "composition": feature_frame(U, "composition"),
    "cm": feature_frame(U, "cm_spectrum", n_jobs=-1),
    "engineered": feature_frame(U, "engineered", n_jobs=-1),
    "groups": atom_blocks[list(GROUP_NAMES)],
    "qeq": atom_blocks[list(QEQ_NAMES)],
    "rdf": atom_blocks[list(RDF_NAMES)],
    "dft_property": mol[DFT_PROPERTIES].copy(),
    "charge": pd.DataFrame([charge_features(q, r) for q, r in zip(mol["q"], mol["R"])],
                           index=mol.index, columns=list(CHARGE_FEATURE_NAMES)),
    "frequency": pd.DataFrame([frequency_features(f) for f in freqs], index=mol.index,
                              columns=list(FREQUENCY_FEATURE_NAMES)),
}
for name, frame in blocks.items():
    frame.index.name = "id"
    assert frame.index.equals(mol.index), name
    print(f"{name:13s} {frame.shape[1]:4d} features  ({'legal' if name in LEGAL_BLOCKS else 'DFT, exploration only'})")
print(f"{ATOM_BLOCK:13s} {atoms_U.X.shape[1]:4d} features per atom, {atoms_U.X.shape[0]:,} atoms (legal)")
print(f"computed in {time.perf_counter() - start:.0f} s")
all_features = pd.concat(blocks.values(), axis=1)
assert all_features.columns.is_unique

### 2a. Is every legal feature invariant?
Rotation, translation and atom relabeling must leave every molecule-level legal feature unchanged (PLAN §8), and
must leave each per-atom row unchanged up to the relabeling itself (row i of the relabeled molecule is the old row
p(i)). 100 random pool molecules; the raw-coordinate negative control must fail.

In [ ]:
from qm9dipole.atoms import describe
from qm9dipole.invariance import TRANSFORMS, invariance_table

sample = U[in_pool].sample(100, random_state=2026)
molecules = list(zip(sample["Z"], sample["R"]))
inv = invariance_table(molecules, seed=2026,
                       descriptors=("composition", "cm_spectrum", "engineered", "groups", "qeq", "rdf", "raw_coordinates"))
rng = np.random.default_rng(2026)
rows = []
for t_name, transform in TRANSFORMS.items():
    worst = 0.0
    for z, r in molecules:
        base = describe(z, r).features
        if t_name == "permutation":
            p = rng.permutation(len(z))
            worst = max(worst, np.abs(describe(z[p], r[p]).features - base[p]).max())
        else:
            worst = max(worst, np.abs(describe(*transform(z, r, rng)).features - base).max())
    rows.append({"descriptor": "atom (per-atom rows)", "transform": t_name, "n_molecules": len(molecules),
                 "mean_change": np.nan, "max_change": worst, "invariant": worst <= 1e-8})
inv = pd.concat([inv, pd.DataFrame(rows)], ignore_index=True)
assert inv.loc[inv["descriptor"] != "raw_coordinates", "invariant"].all()
assert not inv.loc[inv["descriptor"] == "raw_coordinates", "invariant"].any(), "the negative control must fail"
display(inv.style.format({"mean_change": "{:.1e}", "max_change": "{:.1e}"}, na_rep="").hide(axis="index"))

### 2b. Do the new features add information?
Mutual information with |μ| for every legal molecule-level feature outside the Coulomb spectrum (pool, on the EDA's
random subsample), and three dipole estimates side by side: X1's two geometric proxies, the new QEq dipole (all legal),
and the Mulliken point-charge dipole (DFT, exploration only).

In [ ]:
from sklearn.feature_selection import mutual_info_regression

legal_named = pd.concat([blocks[b] for b in ("composition", "engineered", "groups", "qeq", "rdf")], axis=1)
legal_named = legal_named.loc[in_pool, legal_named.loc[in_pool].nunique() > 1]
rs = np.random.default_rng(2026)
sub = np.sort(rs.choice(len(legal_named), size=min(CFG["eda"]["subsample"], len(legal_named)), replace=False))
mi = pd.Series(mutual_info_regression(legal_named.iloc[sub].to_numpy(), y.loc[legal_named.index].iloc[sub].to_numpy(),
                                      random_state=0), index=legal_named.columns)
block_of = {c: b for b, f in blocks.items() for c in f.columns}
top = mi.sort_values(ascending=False).head(20)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(14, 5.6), gridspec_kw={"width_ratios": [1, 1.1]})
new_blocks = {"groups", "qeq", "rdf"}
ax0.barh(top.index[::-1], top.values[::-1], height=0.7,
         color=[SERIES[2] if block_of[c] in new_blocks else ROLE_COLORS["legal"] for c in top.index[::-1]])
ax0.set(xlabel="mutual information with |μ| (nats)", title="Legal features: the 20 most informative (aqua: new in X2)")
ax0.grid(axis="y", visible=False)
dipoles = pd.DataFrame({"bond_dipole_norm\n(X1, legal)": blocks["engineered"]["bond_dipole_norm"],
                        "en_point_dipole\n(X1, legal)": blocks["engineered"]["en_point_dipole"],
                        "qeq_dipole_D\n(X2, legal)": blocks["qeq"]["qeq_dipole_D"],
                        "charge_dipole_D\n(DFT)": blocks["charge"]["charge_dipole_D"]}).loc[in_pool].iloc[sub]
dmi = pd.Series(mutual_info_regression(dipoles.to_numpy(), y.loc[dipoles.index].to_numpy(), random_state=0),
                index=dipoles.columns)
colors = [ROLE_COLORS["legal"], ROLE_COLORS["legal"], SERIES[2], ROLE_COLORS["dft"]]
ax1.bar(range(4), dmi.values, color=colors, width=0.6)
ax1.set_xticks(range(4), dmi.index)
for k, v in enumerate(dmi.values):
    ax1.annotate(f"{v:.2f}", (k, v), xytext=(0, 3), textcoords="offset points", ha="center", color=INK_SECONDARY)
ax1.set(ylabel="mutual information with |μ| (nats)", title="Four dipole estimates: geometry only vs DFT charges")
ax1.grid(axis="x", visible=False)
fig.savefig(FIG_DIR / "explore02_engineered_information.png")
plt.show()
print("mutual information by block (mean over the block's features, pool subsample):")
print(mi.groupby(mi.index.map(block_of)).agg(["mean", "max", "size"]).round(3).to_string())

In [ ]:
g = blocks["groups"].loc[in_pool]
yy = y.loc[g.index]
cases = {"zwitterion (ammonium + carboxylate)": g["zwitterion"] > 0, "carboxylate, no ammonium": (g["n_carboxylate"] > 0) & (g["n_ammonium"] == 0),
         "ammonium, no carboxylate": (g["n_ammonium"] > 0) & (g["n_carboxylate"] == 0), "neither": (g["n_ammonium"] == 0) & (g["n_carboxylate"] == 0)}
display(pd.DataFrame({"molecules": {k: int(m.sum()) for k, m in cases.items()},
                      "median |μ| (D)": {k: yy[m].median() for k, m in cases.items()},
                      "share ≥ 9 D": {k: (yy[m] >= 9).mean() for k, m in cases.items()}})
        .style.format({"median |μ| (D)": "{:.2f}", "share ≥ 9 D": "{:.1%}"}).set_caption("charged groups read from geometry (pool)"))

## 3. Feature-level cleaning (unsupervised, fit on P)

Two rules from `configs/explore.yaml`, applied **within** groups so that a legal feature is never dropped because a
DFT feature duplicates it:
1. **zero variance:** a column that never changes in the pool carries no information;
2. **redundancy:** of two columns with |Spearman ρ| ≥ 0.99 (the same information up to a monotone transform), keep
   the one earlier in priority order: composition → engineered → groups → qeq → rdf; within the DFT group, block
   order; per-atom features in `atoms.ATOM_FEATURE_NAMES` order. **New in X2:** the pair must also reach 0.99 on the
   rows where either column is non-zero. Most atoms have no N, O or F neighbour, so many per-atom features are
   mostly zeros, and those tied zeros alone push ρ over all rows above 0.99 even when the non-zero values carry
   different information (a first dry run of this notebook dropped the directional N and O moments that way).
   True duplicates (bonds_C-F = n_F, U = U0 in rank) still go.

The Coulomb spectrum (`cm`) is exempt from rule 2, as in X1. Per-atom rules run on a random sample of 300,000 pool
atoms. Exact **linear** dependencies (n_H = Σ H-bond counts, one-hot elements that sum to 1) are kept on purpose:
ridge and the network's weight decay handle collinearity, and trees cannot form differences.

In [ ]:
from qm9dipole.atoms import _ranges
from qm9dipole.cleaning import redundant_columns, zero_variance

dropped = {}
threshold = CFG["cleaning"]["redundancy_spearman"]
groups_for_rules = {"legal": ["composition", "engineered", "groups", "qeq", "rdf"], "cm": ["cm"], "DFT": list(DFT_BLOCKS)}
for group, members in groups_for_rules.items():
    pool_frame = pd.concat([blocks[b] for b in members], axis=1).loc[in_pool]
    for col in zero_variance(pool_frame):
        dropped[col] = "zero variance in the pool"
    if group != "cm":
        live = [c for c in pool_frame.columns if c not in dropped]
        for r in redundant_columns(pool_frame[live], threshold).itertuples():
            dropped[r.dropped] = f"redundant with {r.kept} (|ρ| = {r.rho:.4f}; on non-zero rows {r.rho_active:.4f})"

pool_atoms, _ = _ranges(atoms_U.offsets, np.flatnonzero(in_pool))
atom_sample = pd.DataFrame(atoms_U.X[np.sort(rs.choice(pool_atoms, size=min(300_000, len(pool_atoms)), replace=False))],
                           columns=list(ATOM_FEATURE_NAMES)).astype(np.float64)
atom_dropped = {c: "zero variance in the pool" for c in zero_variance(atom_sample)}
live = [c for c in atom_sample.columns if c not in atom_dropped]
for r in redundant_columns(atom_sample[live], threshold).itertuples():
    atom_dropped[r.dropped] = f"redundant with {r.kept} (|ρ| = {r.rho:.4f}; on non-zero rows {r.rho_active:.4f})"

meaning = {row[0]: row[3] for row in DATA_DICTIONARY}
catalog_rows = [{"feature": col, "block": b, "level": "molecule", "legal": b in LEGAL_BLOCKS, "kept": col not in dropped,
                 "reason": dropped.get(col, ""), "description": meaning.get(col, "")}
                for b, frame in blocks.items() for col in frame.columns]
catalog_rows += [{"feature": col, "block": ATOM_BLOCK, "level": "atom", "legal": True, "kept": col not in atom_dropped,
                  "reason": atom_dropped.get(col, ""), "description": "per-atom environment (atoms.py)"}
                 for col in ATOM_FEATURE_NAMES]
catalog = pd.DataFrame(catalog_rows)
summary = catalog.groupby("block", sort=False).agg(features=("feature", "size"), kept=("kept", "sum"))
display(summary.T)
shown = catalog[~catalog["kept"]][["feature", "block", "reason"]]
print(f"dropped {len(shown)} of {len(catalog)} features; first rows by block:")
display(shown.groupby("block").head(6).style.hide(axis="index"))

## 4. Sanity checks

In [ ]:
features = all_features[catalog.loc[catalog["kept"] & (catalog["level"] == "molecule"), "feature"]].astype(np.float32)
assert np.isfinite(features.to_numpy()).all(), "NaN or inf in the feature table"
assert np.isfinite(atoms_U.X).all(), "NaN or inf in the per-atom features"
needed = set(np.concatenate([o for o in splits.order.values()])) | set(splits.dev) | set(splits.dev_unseen)
assert needed <= set(features.index), "a training or development molecule is missing"
assert features.index.is_unique
assert not catalog.loc[catalog["block"].isin(DFT_BLOCKS), "legal"].any(), "a DFT feature is marked legal"
sets = feature_sets(catalog)
for name, cols in sets.items():
    legal = catalog.set_index("feature").loc[cols, "legal"].all()
    print(f"{name:15s} {len(cols):4d} features  {'headline-legal' if legal else 'EXPLORATION ONLY (uses DFT outputs)'}")
assert all(catalog.set_index("feature").loc[sets[s], "legal"].all()
           for s in ("composition", "cm", "structure", "structure+", "all_legal"))
kept_atoms = catalog.loc[(catalog["level"] == "atom") & catalog["kept"], "feature"].tolist()
print(f"per-atom features kept: {len(kept_atoms)} of {len(ATOM_FEATURE_NAMES)}")
print("no NaN/inf; every training and development molecule present; DFT features never in a legal set")

## 5. Save for the next notebooks

In [ ]:
out = features.join(mol[["mu", "formula", "n_heavy"]]).join(subset)
FEATURES_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_parquet(FEATURES_PATH)
_ = save_result(catalog, CATALOG_NAME, config=CFG["cleaning"], n_molecules=len(out), n_atoms=int(atoms_U.X.shape[0]),
                label="exploratory; feature catalog (molecule and atom level) for the development universe")
print(f"wrote {FEATURES_PATH.relative_to(REPO_ROOT).as_posix()}: {out.shape[0]:,} molecules × {features.shape[1]} features "
      f"({FEATURES_PATH.stat().st_size / 1e6:.0f} MB)")
print(f"wrote results/{CATALOG_NAME}.csv: {len(catalog)} features, {catalog['kept'].sum()} kept")

## Summary

1. **Record-level cleaning.** 277 doubled frequency lists in the development universe (255 in the pool) were cut to
   one copy. Moments of inertia from Z and R reproduce QM9's rotational constants to 1.4 × 10⁻⁵ and replace them.
   Duplicates were already removed before splitting.
2. **Features.** Six legal molecule-level blocks (composition 5, Coulomb spectrum 29, X1's engineered 33, and new:
   charged groups and atom types 21, QEq 3, radial distributions 120), three DFT blocks for exploration only, and
   **93 features for each of 1,939,514 atoms**, all from Z and R, in about 75 s.
3. **Every legal feature is invariant** to rotation, translation and relabeling, to ≤ 9 × 10⁻¹³ (per-atom rows: ≤
   3 × 10⁻¹⁴, permuted with the atoms); the raw-coordinate negative control changes by 5–10 Å and fails, as it must.
4. **The new features carry the most information.** The QEq dipole is the most informative legal feature (MI 0.52
   nats), ahead of X1's best (total bond polarity 0.38, bond-dipole sum 0.30), and 13 of the 20 most informative
   legal features are new (QEq, atom types, radial-distribution bins of C–O, C–N and C–H). The DFT point-charge
   dipole (1.05 nats) shows how much a single number that knows the electron density still adds.
5. **Charged groups read from geometry mark the tail.** Of 128 zwitterions (ammonium + carboxylate), 99% are at
   ≥ 9 D; molecules with only a carboxylate (36) or only an ammonium N (40) are at ≥ 9 D in 61% and 83% of cases.
   Together they account for about 182 of the pool's 223 molecules at ≥ 9 D; among the other 98,994 molecules only
   0.04% reach 9 D.
6. **Cleaning kept 288 of 323 features.** The redundancy rule now also requires ρ ≥ 0.99 on the rows where either
   feature is non-zero: tied zeros of sparse features had pushed ρ over all rows past 0.99 and, in a first dry run,
   dropped the directional N and O moments of each atom. Dropped now: bonds_C-F (= n_F) and six bond types that never
   occur, atoms_C1 (never occurs), n_ammonium (= atoms_N4), 14 radial-distribution bins equal to a neighbour bin or a
   bond count, U/H/G (= U0 in rank), the X–H stretch count (≈ zpve), and 8 per-atom near-bond radial and moment
   features that are monotone in another where they vary.
7. **Feature sets for notebooks 03 → 05:** composition (5), cm (29), structure (31, X1's best), structure+ (53) and
   all_legal (188), all headline-legal; structure+dft (46), exploration only; and 85 per-atom features for the
   latent-charge model.

**Carried forward:** notebook 03 re-chooses the scaling and target transform for these sets at small N (Track B) and
large N (Track A), and chooses the reduction method and the number of inputs for the quantum-comparable models.